In [1]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver, gold
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

DataFrame[]

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [ ]:
bronze_instance.load().transform().write(mode="overwrite").execute("people")

2025-02-27 16:45:31 | people | execute | Started
2025-02-27 16:45:31 | people | load | Started
2025-02-27 16:45:38 | people | load | Completed in 0.1 min
2025-02-27 16:45:38 | people | transform | Started
2025-02-27 16:45:38 | people | transform | Completed in 0.0 min
2025-02-27 16:45:38 | people | write | Started
2025-02-27 16:46:39 | people | write | Completed in 1.02 min
2025-02-27 16:46:40 | people | execute | Completed in 1.13 min


In [10]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-02-27 16:45:...|        Cliegg Lars| 62|https://www.swapi...|{"created": "2025...|
|2025-02-27 16:45:...|  Poggle the Lesser| 63|https://www.swapi...|{"created": "2025...|
|2025-02-27 16:45:...|    Luminara Unduli| 64|https://www.swapi...|{"created": "2025...|
|2025-02-27 16:45:...|      Barriss Offee| 65|https://www.swapi...|{"created": "2025...|
|2025-02-27 16:45:...|              Dormé| 66|https://www.swapi...|{"created": "2025...|
|2025-02-27 16:45:...|              Dooku| 67|https://www.swapi...|{"created": "2025...|
|2025-02-27 16:45:...|Bail Prestor Organa| 68|https://www.swapi...|{"created": "2025...|
|2025-02-27 16:45:...|         Jango Fett| 69|https://www.swapi...|{"created": "2025...|
|2025-02

# 2 Silver

In [11]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [12]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [13]:
class StarWarsSilver(silver.Silver):
    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        if table == "people":
            df = self.transf_people(df)
        return df

    def transf_people(self, df: DataFrame) -> DataFrame:
        df = (
            df.withColumn("height", df.properties.height)
            .withColumn("mass", df.properties.mass)
            .withColumn("gender", df.properties.gender)
            .drop("url", "properties")
        )
        return df


silver_instance = StarWarsSilver(spark, **options)

In [14]:
silver_instance.load().transform().write(mode="overwrite", merge_schema=True).execute(
    "people"
)

2025-02-27 16:46:46 | people | execute | Started
2025-02-27 16:46:46 | people | load | Started
2025-02-27 16:46:46 | people | load | Completed in 0.0 min
2025-02-27 16:46:46 | people | transform | Started
2025-02-27 16:46:46 | people | transform | Completed in 0.0 min
2025-02-27 16:46:46 | people | write | Started
2025-02-27 16:46:51 | people | write | Completed in 0.07 min
2025-02-27 16:46:51 | people | execute | Completed in 0.07 min


In [15]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 82
+--------------------------+--------------------------+---------------------+---+-------+-------+-------------+
|LH_SilverTS               |LH_BronzeTS               |name                 |uid|height |mass   |gender       |
+--------------------------+--------------------------+---------------------+---+-------+-------+-------------+
|2025-02-27 16:46:47.033492|2025-02-27 16:45:39.931138|Cliegg Lars          |62 |183    |unknown|male         |
|2025-02-27 16:46:47.033492|2025-02-27 16:45:39.931138|Poggle the Lesser    |63 |183    |80     |male         |
|2025-02-27 16:46:47.033492|2025-02-27 16:45:39.931138|Luminara Unduli      |64 |170    |56.2   |female       |
|2025-02-27 16:46:47.033492|2025-02-27 16:45:39.931138|Barriss Offee        |65 |166    |50     |female       |
|2025-02-27 16:46:47.033492|2025-02-27 16:45:39.931138|Dormé                |66 |165    |unknown|female       |
|2025-02-27 16:46:47.033492|2025-02-27 16:45:39.931138|Dooku                |67 |193    |80

# 3 Gold

In [16]:
options = {
    "catalog": CATALOG,
    "source_schema": "silver",
    "target_schema": "gold",
}

In [17]:
class StarWarsGold(gold.Gold):
    def people_per_gender(self, df: DataFrame, table: str) -> DataFrame:
        df = df.where("gender <> 'n/a'")
        df = df.where("gender <> 'none'")
        df = df.groupBy("gender").count()
        return df

    def all_females(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("gender = 'female'").drop("LH_SilverTS", "LH_BronzeTS")


gold_instance = StarWarsGold(spark, **options)

In [18]:
gold_instance.load(source_tbl="people").transform(
    tbl_transformations={
        "peoplegender": "people_per_gender",
        "peoplefemale": "all_females",
    }
).write(mode="overwrite", merge_schema=True).execute("peoplegender", "peoplefemale")

2025-02-27 16:46:54 | peoplegender | execute | Started
2025-02-27 16:46:54 | peoplegender | load | Started
2025-02-27 16:46:54 | peoplegender | load | Completed in 0.0 min
2025-02-27 16:46:54 | peoplegender | transform | Started
2025-02-27 16:46:54 | peoplegender | transform | Completed in 0.0 min
2025-02-27 16:46:54 | peoplegender | write | Started
2025-02-27 16:46:59 | peoplegender | write | Completed in 0.08 min
2025-02-27 16:46:59 | peoplegender | execute | Completed in 0.08 min
2025-02-27 16:46:59 | peoplefemale | execute | Started
2025-02-27 16:46:59 | peoplefemale | load | Started
2025-02-27 16:46:59 | peoplefemale | load | Completed in 0.0 min
2025-02-27 16:46:59 | peoplefemale | transform | Started
2025-02-27 16:46:59 | peoplefemale | transform | Completed in 0.0 min
2025-02-27 16:46:59 | peoplefemale | write | Started
2025-02-27 16:47:03 | peoplefemale | write | Completed in 0.05 min
2025-02-27 16:47:03 | peoplefemale | execute | Completed in 0.05 min


In [19]:
df = spark.sql(f"SELECT * FROM {CATALOG}.gold.peoplegender")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 3
+--------------------------+-------------+-----+
|LH_GoldTS                 |gender       |count|
+--------------------------+-------------+-----+
|2025-02-27 16:46:54.385114|female       |17   |
|2025-02-27 16:46:54.385114|male         |60   |
|2025-02-27 16:46:54.385114|hermaphrodite|1    |
+--------------------------+-------------+-----+



In [20]:
df = spark.sql(f"SELECT * FROM {CATALOG}.gold.peoplefemale")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 17
+--------------------------+------------------+---+------+-------+------+
|LH_GoldTS                 |name              |uid|height|mass   |gender|
+--------------------------+------------------+---+------+-------+------+
|2025-02-27 16:46:59.981365|Luminara Unduli   |64 |170   |56.2   |female|
|2025-02-27 16:46:59.981365|Barriss Offee     |65 |166   |50     |female|
|2025-02-27 16:46:59.981365|Dormé             |66 |165   |unknown|female|
|2025-02-27 16:46:59.981365|Zam Wesell        |70 |168   |55     |female|
|2025-02-27 16:46:59.981365|Taun We           |73 |213   |unknown|female|
|2025-02-27 16:46:59.981365|Jocasta Nu        |74 |167   |unknown|female|
|2025-02-27 16:46:59.981365|R4-P17            |75 |96    |unknown|female|
|2025-02-27 16:46:59.981365|Shaak Ti          |78 |178   |57     |female|
|2025-02-27 16:46:59.981365|Sly Moore         |82 |178   |48     |female|
|2025-02-27 16:46:59.981365|Shmi Skywalker    |43 |163   |unknown|female|
|2025-02-27 16:46:59.9813

# 4 Clean Up

In [21]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.gold CASCADE")

DataFrame[]